# Analysis

**Hypothesis**: The interferon-stimulated gene (ISG) signature is broadly upregulated across multiple immune cell types in severe COVID-19, with the most pronounced increase in monocytes and pDCs.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("example/covid19.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


# Analysis Plan

**Hypothesis**: The interferon-stimulated gene (ISG) signature is broadly upregulated across multiple immune cell types in severe COVID-19, with the most pronounced increase in monocytes and pDCs.

## Steps:
- Define a robust list of interferon-stimulated genes (ISGs) and compute an ISG expression score per cell using scanpy's score_genes function with a fixed random seed.
- For each coarse cell type, subset cells and extract ISG scores for COVID and Healthy groups, then perform Mann-Whitney U tests (two-sided) and correct for multiple comparisons using Bonferroni correction.
- Create a summary violin plot showing ISG scores by cell type and condition, with statistical significance annotations (p-values or asterisks) for each cell type.
- Print the corrected p-values and interpret which cell types show the most significant upregulation, with special attention to monocytes and pDCs.


## This code imports necessary libraries, defines an expanded ISG gene list, filters to those present in the dataset, and computes an ISG expression score per cell using scanpy's score_genes function with a fixed random seed (42) for reproducibility.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# Define ISG gene list (expand with additional common ISGs)
isg_genes = ['IFIT1', 'IFIT2', 'IFIT3', 'MX1', 'ISG15', 'OAS1', 'OAS2', 'OAS3', 'IFI44L', 'RSAD2', 'MX2']
# Filter to genes that exist in the dataset
valid_isg = [g for g in isg_genes if g in adata.var_names]
print(f"Using ISG genes: {valid_isg}")

# Compute ISG score per cell with fixed random seed for reproducibility
sc.tl.score_genes(adata, gene_list=valid_isg, score_name='ISG_score', ctrl_size=50, random_state=42)
print("ISG score computed and stored in adata.obs['ISG_score']")

### Agent Interpretation

No results found

## Next Steps
Step 1: For each coarse cell type, subset cells and extract ISG scores for COVID and Healthy groups, then perform Mann-Whitney U tests (two-sided) and correct for multiple comparisons using Bonferroni correction.
Step 2: Create a summary violin plot showing ISG scores by cell type and condition, with statistical significance annotations (p-values or asterisks) for each cell type.
Step 3: Print the corrected p-values and interpret which cell types show the most significant upregulation, with special attention to monocytes and pDCs.

## This code performs two-sided Mann-Whitney U tests for each coarse cell type comparing ISG scores between COVID and Healthy groups, applies Bonferroni correction for multiple comparisons, stores the corrected p-values, and prints them.

In [ ]:
# Step 1: Perform Mann-Whitney U tests per cell type comparing ISG scores between COVID and Healthy
from scipy.stats import mannwhitneyu

# Ensure ISG_score exists in adata.obs
if 'ISG_score' not in adata.obs:
    raise ValueError("ISG_score not found. Please run ISG scoring first.")

# Get list of coarse cell types
cell_types = adata.obs['cell_type_coarse'].unique()
p_values = {}
for ct in cell_types:
    mask = adata.obs['cell_type_coarse'] == ct
    scores_covid = adata[mask & (adata.obs['Status'] == 'COVID')].obs['ISG_score'].values
    scores_healthy = adata[mask & (adata.obs['Status'] == 'Healthy')].obs['ISG_score'].values
    if len(scores_covid) > 0 and len(scores_healthy) > 0:
        stat, p = mannwhitneyu(scores_covid, scores_healthy, alternative='two-sided')
        p_values[ct] = p
    else:
        p_values[ct] = np.nan

# Bonferroni correction: multiply by number of tests (cell types with non-NaN p-value)
valid_cell_types = [ct for ct, p in p_values.items() if not np.isnan(p)]
num_tests = len(valid_cell_types)
bonferroni_corrected = {ct: min(p_values[ct] * num_tests, 1.0) for ct in valid_cell_types}
print('Corrected p-values (Bonferroni):')
for ct in sorted(bonferroni_corrected.keys()):
    print(f'{ct}: {bonferroni_corrected[ct]:.4e}')
# Store results for later steps
adata.uns['isg_test_results'] = bonferroni_corrected

### Agent Interpretation

No results found

## Next Steps
Step 1: For each donor, compute the mean expression of inflammatory ligands (TNF, IL1B, IL6) in monocytes and receptors (TNFRSF1A, IL1R1, IL6R) in T cells. Create an average ligand score per donor for monocytes and an average receptor score per donor for T cells, stratified by condition.
Step 2: Perform Spearman correlation between monocyte ligand score and T cell receptor score across donors, separately for COVID and Healthy. Visualize with a scatter plot and report correlation coefficients and p-values for each condition.

## This step computes per-donor average expression of inflammatory ligands (TNF, IL1B, IL6) in monocytes and receptors (TNFRSF1A, IL1R1, IL6R) in T cells, then averages across genes to obtain a single ligand score and receptor score per donor, stratified by condition (COVID vs Healthy). The merged dataframe is saved for correlation analysis.

In [ ]:
import numpy as np
import pandas as pd

# Define key inflammatory ligands and their receptors
ligands = ['TNF', 'IL1B', 'IL6']
receptors = ['TNFRSF1A', 'IL1R1', 'IL6R']

# Keep only genes present in adata
valid_ligands = [g for g in ligands if g in adata.var_names]
valid_receptors = [g for g in receptors if g in adata.var_names]
print(f'Valid ligands: {valid_ligands}')
print(f'Valid receptors: {valid_receptors}')

# Define monocyte and T cell types
monocyte_types = ['CD14 Monocyte', 'CD16 Monocyte']
t_cell_types = ['CD4 T', 'CD8 T', 'gd T']

# Subset adata
mono_mask = adata.obs['cell_type_coarse'].isin(monocyte_types)
tcell_mask = adata.obs['cell_type_coarse'].isin(t_cell_types)

# Get expression matrices (may be sparse; use toarray() if needed)
mono_expr = adata[mono_mask, valid_ligands].X
if hasattr(mono_expr, 'toarray'):
    mono_expr = mono_expr.toarray()
tcell_expr = adata[tcell_mask, valid_receptors].X
if hasattr(tcell_expr, 'toarray'):
    tcell_expr = tcell_expr.toarray()

# Create DataFrames with donor and condition info
mono_df = pd.DataFrame(mono_expr, columns=valid_ligands,
                       index=adata.obs[mono_mask].index)
mono_df['Donor_full'] = adata.obs[mono_mask]['Donor_full'].values
mono_df['Status'] = adata.obs[mono_mask]['Status'].values

tcell_df = pd.DataFrame(tcell_expr, columns=valid_receptors,
                        index=adata.obs[tcell_mask].index)
tcell_df['Donor_full'] = adata.obs[tcell_mask]['Donor_full'].values
tcell_df['Status'] = adata.obs[tcell_mask]['Status'].values

# Aggregate per donor: mean ligand score for monocytes, mean receptor score for T cells
mono_donor = mono_df.groupby(['Donor_full', 'Status'])[valid_ligands].mean()
mono_donor['LigandScore'] = mono_donor[valid_ligands].mean(axis=1)
tcell_donor = tcell_df.groupby(['Donor_full', 'Status'])[valid_receptors].mean()
tcell_donor['ReceptorScore'] = tcell_donor[valid_receptors].mean(axis=1)

# Merge on donor and status
merged = mono_donor[['LigandScore']].join(tcell_donor[['ReceptorScore']], how='inner')
merged.reset_index(inplace=True)
# merged DataFrame will be used in step 2

### Agent Interpretation

No results found

## Next Steps
Step 1: Perform Spearman correlation between monocyte ligand score and T cell receptor score across donors for each condition separately, compare correlations using Fisher’s z-test, and visualize with a scatter plot including regression lines and annotated statistics.

## Cleans merged data, computes Spearman correlations for COVID and Healthy separately, performs a Fisher z-test to compare them, and creates a scatter plot with regression lines and annotated statistics to visually assess the relationship.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Clean merged data
merged_clean = merged.dropna(subset=['LigandScore', 'ReceptorScore'])
covid = merged_clean[merged_clean['Status'] == 'COVID']
healthy = merged_clean[merged_clean['Status'] == 'Healthy']

# Spearman correlations
corr_c, p_c = spearmanr(covid['LigandScore'], covid['ReceptorScore'])
corr_h, p_h = spearmanr(healthy['LigandScore'], healthy['ReceptorScore'])
n_c = len(covid)
n_h = len(healthy)

# Fisher z-test for comparing two correlations
z_c = np.arctanh(corr_c)
z_h = np.arctanh(corr_h)
se = np.sqrt(1/(n_c - 3) + 1/(n_h - 3))
z_stat = (z_c - z_h) / se
p_comp = 2 * (1 - stats.norm.cdf(abs(z_stat)))  # two-sided

# Scatter plot with regression lines
plt.figure(figsize=(8,6))
sns.regplot(x='LigandScore', y='ReceptorScore', data=covid, color='red', label=f'COVID (n={n_c})', ci=None, line_kws={"lw":2, "ls":"--"})
sns.regplot(x='LigandScore', y='ReceptorScore', data=healthy, color='blue', label=f'Healthy (n={n_h})', ci=None, line_kws={"lw":2, "ls":"--"})
plt.xlabel('Monocyte Ligand Score')
plt.ylabel('T Cell Receptor Score')
plt.title('Monocyte Ligand vs T Cell Receptor Score by Condition')
plt.legend()
plt.grid(True, alpha=0.3)

# Annotate correlation coefficients and p-values on plot
text_c = f'COVID: r={corr_c:.3f}, p={p_c:.3e}'
text_h = f'Healthy: r={corr_h:.3f}, p={p_h:.3e}'
text_comp = f'Correlation comparison (z-test): p={p_comp:.4e}'
plt.text(0.95, 0.95, text_c, transform=plt.gca().transAxes, ha='right', va='top', color='red', fontsize=9)
plt.text(0.95, 0.88, text_h, transform=plt.gca().transAxes, ha='right', va='top', color='blue', fontsize=9)
plt.text(0.95, 0.81, text_comp, transform=plt.gca().transAxes, ha='right', va='top', color='black', fontsize=9, fontstyle='italic')
plt.tight_layout()
plt.show()

# Print results
print(f'COVID: Spearman r = {corr_c:.3f}, p-value = {p_c:.4e}')
print(f'Healthy: Spearman r = {corr_h:.3f}, p-value = {p_h:.4e}')
print(f'Comparison (Fisher z-test): z = {z_stat:.3f}, p = {p_comp:.4e}')
if p_c < 0.05:
    print('COVID correlation is statistically significant.')
else:
    print('COVID correlation is not statistically significant.')
if p_h < 0.05:
    print('Healthy correlation is statistically significant.')
else:
    print('Healthy correlation is not statistically significant.')
if p_comp < 0.05:
    print('The two correlations are significantly different.')
else:
    print('The two correlations are not significantly different.')

### Agent Interpretation

No results found